# HuggingFace Transformers — Practice

Write your own code in the empty cell under each task. Do not look at the reference notebook until you are done.

## Part 1 — Transformers Pipelines

### 1. Import whatever you need to build a task pipeline from the transformers library.

In [1]:
from  transformers import pipeline, AutoTokenizer

### 2. Create a sentiment analysis pipeline without naming a model. Read the warnings it prints and write down, in a markdown cell you add yourself, which model and revision it fell back to.

In [13]:
sentiment_analysis = pipeline('sentiment-analysis') 

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

### 3. Run that pipeline on the sentence: I'm so excited to be learning about large language models

In [14]:
sentiment_analysis("I'm so excited to be learning about large language models")

[{'label': 'POSITIVE', 'score': 0.9997096657752991}]

### 4. Run the same pipeline on a sentence you expect to score as negative, and on one you expect to be borderline. Compare the scores.

In [15]:
sentiment_analysis('I do not like this movie at all')


[{'label': 'NEGATIVE', 'score': 0.9957866072654724}]

In [16]:
sentiment_analysis('The movie was released last year.')

[{'label': 'POSITIVE', 'score': 0.9356309771537781}]

### 5. Pass a list of three sentences to the pipeline in a single call. Describe the shape of what comes back.

In [17]:
sentences = [
    "I am learning how to use Hugging Face transformers.",
    "The model predicted this sentence as positive.",
    "The weather is calm and the sky is clear today.",
]

sentiment_analysis(sentences)

[{'label': 'POSITIVE', 'score': 0.9412967562675476},
 {'label': 'POSITIVE', 'score': 0.9210985898971558},
 {'label': 'POSITIVE', 'score': 0.9996886253356934}]

### 6. Build a named entity recognition pipeline using the model dslim/bert-base-NER.

In [18]:
ner_tagging = pipeline("ner", model = "dslim/bert-base-NER")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 7. Run it on: Her name is Anna and she works in New York City for Morgan Stanley

In [19]:
ner_tags =  ner_tagging('Run it on: Her name is Anna and she works in New York City for Morgan Stanley')
print(ner_tags)

[{'entity': 'B-PER', 'score': np.float32(0.9947484), 'index': 8, 'word': 'Anna', 'start': 23, 'end': 27}, {'entity': 'B-LOC', 'score': np.float32(0.9995988), 'index': 13, 'word': 'New', 'start': 45, 'end': 48}, {'entity': 'I-LOC', 'score': np.float32(0.99932826), 'index': 14, 'word': 'York', 'start': 49, 'end': 53}, {'entity': 'I-LOC', 'score': np.float32(0.9995896), 'index': 15, 'word': 'City', 'start': 54, 'end': 58}, {'entity': 'B-ORG', 'score': np.float32(0.99690825), 'index': 17, 'word': 'Morgan', 'start': 63, 'end': 69}, {'entity': 'I-ORG', 'score': np.float32(0.9984659), 'index': 18, 'word': 'Stanley', 'start': 70, 'end': 77}]


### 8. From that output, print only the entity text and its entity type for each result.

In [20]:
for obj in ner_tags:
    print("{:} - {:}".format(obj["entity"], obj["word"]))

B-PER - Anna
B-LOC - New
I-LOC - York
I-LOC - City
B-ORG - Morgan
I-ORG - Stanley


### 9. Rebuild the NER pipeline so that word pieces belonging to the same entity are merged into one result, then rerun the same sentence.

In [21]:
ner_tagging = pipeline("ner", model = "dslim/bert-base-NER", aggregation_strategy="simple")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
ner_tags = ner_tagging("Her name is Anna and she works in New York City for Morgan Stanley")
print(ner_tags)

[{'entity_group': 'PER', 'score': np.float32(0.9954881), 'word': 'Anna', 'start': 12, 'end': 16}, {'entity_group': 'LOC', 'score': np.float32(0.9995275), 'word': 'New York City', 'start': 34, 'end': 47}, {'entity_group': 'ORG', 'score': np.float32(0.99684036), 'word': 'Morgan Stanley', 'start': 52, 'end': 66}]


### 10. Build a zero-shot classification pipeline using facebook/bart-large-mnli.

In [23]:
zero_shot_classification = pipeline("zero-shot-classification", model = "facebook/bart-large-mnli")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

### 11. Classify the sequence 'one day I will see the world' against the candidate labels travel, cooking, dancing.

In [24]:
zero_shot_classification("one day I will see the world", candidate_labels=["travel", "cooking", "dancing"])

{'sequence': 'one day I will see the world',
 'labels': ['travel', 'dancing', 'cooking'],
 'scores': [0.9938651919364929, 0.0032738002482801676, 0.002861030399799347]}

### 12. Classify the same sequence against a different set of labels of your own choosing. Note what happens to the scores.

In [25]:
zero_shot_classification("one day I will see the world", candidate_labels = ["studying", "eating", "praying"])

{'sequence': 'one day I will see the world',
 'labels': ['studying', 'praying', 'eating'],
 'scores': [0.4673256576061249, 0.4492185115814209, 0.08345581591129303]}

### 13. Classify a sequence that fits two labels at once, and rerun it allowing more than one label to be true.

In [26]:
sentence = "I need a refund because my new headphones arrived broken."
candidate_labels = ["refund request", "product complaint"]

zero_shot_classification(sentence, candidate_labels)

{'sequence': 'I need a refund because my new headphones arrived broken.',
 'labels': ['refund request', 'product complaint'],
 'scores': [0.6444340348243713, 0.35556599497795105]}

## Part 2 — Pre-trained Tokenizers

### 1. Load the tokenizer for bert-base-uncased.

In [27]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

### 2. Call the tokenizer directly on the sentence: I'm so excited to be learning about large language models. Print the full result.

In [28]:
input =  tokenizer("I'm so excited to be learning about large language models")

### 3. In a markdown cell you add yourself, state what each key in that result holds and why the id list is longer than the number of words in the sentence.

input_ids = tokens in vocab
token_type_ids = belongs to which sentence 
attention_mask = real vs padded bits

### 4. Produce the token strings for the same sentence without calling the tokenizer directly.

In [29]:
tokens = tokenizer.convert_ids_to_tokens(input['input_ids'])
print(tokens)

['[CLS]', 'i', "'", 'm', 'so', 'excited', 'to', 'be', 'learning', 'about', 'large', 'language', 'models', '[SEP]']


### 5. Convert those token strings into their ids.

In [30]:
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(token_ids)

[101, 1045, 1005, 1049, 2061, 7568, 2000, 2022, 4083, 2055, 2312, 2653, 4275, 102]


### 6. Compare that id list against the input_ids from task 2 and explain the difference.

### 7. Decode the id list back into text. Note anything that did not survive the round trip.

In [31]:
tokens = tokenizer.decode(token_ids)
print(tokens)

[CLS] i ' m so excited to be learning about large language models [SEP]


### 8. Decode the ids 101 and 102 individually.

In [32]:
print(tokenizer.decode(101)) 
print(tokenizer.decode(102)) 


[CLS]
[SEP]


### 9. Load the tokenizer for xlnet-base-cased.

In [33]:
tokenizer2 = AutoTokenizer.from_pretrained('xlnet-base-cased')

### 10. Tokenize the same sentence with it and print both the full output and the token strings.

In [34]:
input2 = tokenizer2("I'm so excited to be learning about large language models.")
print(input2)

{'input_ids': [35, 26, 98, 102, 5564, 22, 39, 1899, 75, 392, 1243, 2626, 9, 4, 3], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


### 11. Decode the ids 3 and 4 with this tokenizer.

In [35]:
print(tokenizer2.decode(3))
print(tokenizer2.decode(4))

<cls>
<sep>


### 12. Write down, in a markdown cell, three concrete differences between how the two tokenizers handle this sentence.

### 13. Tokenize a word neither tokenizer is likely to have in its vocabulary with both tokenizers and compare the pieces.

In [36]:
print(tokenizer.tokenize("usama"))
print(tokenizer2.tokenize("usama"))

['usa', '##ma']
['▁us', 'ama']


### 14. Tokenize two sentences at once with padding and truncation turned on, with a maximum length you choose, and inspect the attention mask.

In [37]:
sentences = "I'm so excited to be learning about large language models. Tokenizers split text into pieces that language models can understand."

batch = tokenizer(
    sentences,
    padding=True,
    truncation=True,
)

print(batch)
print("Attention mask:")
print(batch["attention_mask"])

{'input_ids': [101, 1045, 1005, 1049, 2061, 7568, 2000, 2022, 4083, 2055, 2312, 2653, 4275, 1012, 19204, 17629, 2015, 3975, 3793, 2046, 4109, 2008, 2653, 4275, 2064, 3305, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
Attention mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## Part 3 — Running a Model Manually with PyTorch

### 1. Import the tokenizer class, the sequence classification model class, and torch.

In [38]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

### 2. Load the tokenizer for distilbert-base-uncased-finetuned-sst-2-english.

In [39]:
model = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model)

### 3. Tokenize the sentence 'I'm so excited to be learning about large language models' so that the result comes back as PyTorch tensors. Print it.

In [40]:
input = tokenizer(
    "I'm so excited to be learning about large language models",
    return_tensors = 'pt',
)

print(
    input
) 

{'input_ids': tensor([[ 101, 1045, 1005, 1049, 2061, 7568, 2000, 2022, 4083, 2055, 2312, 2653,
         4275,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


### 4. State, in a markdown cell you add yourself, how this output differs from the plain Python output in the previous notebook.

it adds a tensor list which is sutiable for AI

### 5. Load the matching sequence classification model.

In [41]:
classification_model = AutoModelForSequenceClassification.from_pretrained(model)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

### 6. Run the tokenized input through the model without tracking gradients, and keep the logits.

In [42]:
with torch.no_grad():
    logits = classification_model(**input).logits

### 7. Print the raw logits and their shape.

In [43]:
print(logits)

tensor([[-3.9386,  4.2055]])


### 8. Find the index of the highest logit as a plain Python integer.

In [44]:
output =  logits.argmax().item()
output

1

### 9. Turn that index into a human readable label using the model's own configuration.

In [45]:
classification_model.config.id2label[output]

'POSITIVE'

### 10. Convert the logits into probabilities and print the probability for each label.

In [46]:
probablities =  torch.softmax(logits, dim = 1)
print(probablities)

tensor([[2.9035e-04, 9.9971e-01]])


### 11. Check that the label you got matches what the sentiment pipeline returned in notebook 1.

In [47]:
print(sentiment_analysis("I'm so excited to be learning about large language models"))

[{'label': 'POSITIVE', 'score': 0.9997096657752991}]


### 12. Repeat the whole flow for a batch of three sentences in one forward pass, and report the predicted label and confidence for each.

In [103]:
sentences = [
    "I’m here to help you.",
    "I can answer questions, explain concepts, and work through code.",
    "Tell me what you want to do next.",
]

inputs = tokenizer2(sentences, return_tensors="pt", padding=True, truncation=True)
print(input)

{'input_ids': tensor([[ 101, 1045, 2064, 3437, 3980, 1010, 4863, 8474, 1010, 1998, 2147, 2083,
         3642, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [104]:
with torch.no_grad():
    logits = classification_model(**inputs).logits

print(logits)

tensor([[ 1.1633, -1.0605],
        [ 1.3322, -1.1934],
        [ 0.9859, -0.9012]])


In [105]:
score = logits.argmax(dim = -1) # for three seperate labels
print(score)
labels = [classification_model.config.id2label[i.item()] for i in score]
print(labels)

tensor([0, 0, 0])
['NEGATIVE', 'NEGATIVE', 'NEGATIVE']


## Part 4 — Saving and Loading Models

### 1. Load a tokenizer and a sequence classification model of your choice into memory.

In [88]:
tokenizer = AutoTokenizer.from_pretrained(model)
classification_model = AutoModelForSequenceClassification.from_pretrained(model)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

### 2. Choose a directory name to save them into and assign it to a variable.

In [89]:
model_directory = 'my_saved_model'

### 3. Save the tokenizer to that directory and print what the call returns.

In [90]:
tokenizer.save_pretrained(model_directory)
classification_model.save_pretrained(model_directory)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### 4. Save the model to that same directory.

### 5. List the files that ended up in the directory and write down, in a markdown cell you add yourself, what each one is for.

### 6. Load the tokenizer back from the directory into a new variable.

In [91]:
my_tokenizer = AutoTokenizer.from_pretrained(model_directory)

### 7. Load the model back from the directory into a new variable.

In [92]:
my_model = AutoModelForSequenceClassification.from_pretrained(model_directory)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

### 8. Run a sentence through the reloaded tokenizer and model and confirm you get the same prediction as before saving.

In [98]:
input = tokenizer("I can answer questions, explain concepts, and work through code.", return_tensors= 'pt')
print(input)

with torch.no_grad():
    logits = my_model(**input).logits

print(logits)

score = logits.argmax().item()
print(score)

my_model.config.id2label[score]

{'input_ids': tensor([[ 101, 1045, 2064, 3437, 3980, 1010, 4863, 8474, 1010, 1998, 2147, 2083,
         3642, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
tensor([[-1.3343,  1.3968]])
1


'POSITIVE'

### 9. Build a pipeline that uses your reloaded model and tokenizer instead of downloading anything, and run it on the same sentence.

In [108]:
sentiment_analysis = pipeline('sentiment-analysis', model = my_model, tokenizer = my_tokenizer)
sentiment_analysis("I can answer questions, explain concepts, and work through code.")

[{'label': 'POSITIVE', 'score': 0.9388313889503479}]

### 10. Explain, in a markdown cell, what would happen if you saved the model but not the tokenizer and then tried to reload both from that directory.